### Parabolic integration error

Given three points in the Cartesian $xy$-plane, we want to:
- Compute the interpolating parabola;
- Compute the integral;
- Estimate the error of the integral (both inherited and algorithmic).

In [ ]:
import sympy as smp  # Only SymPy

In [ ]:
x1, x2, x3, y1, y2, y3 = smp.symbols("x_1, x_2, x_3, y_1, y_2, y_3", real = True, constant = True) # Points (x_1, y_1), (x_2, y_2), (x3, y_3)
h = smp.symbols("h", real = True, positive = True, constant = True) # Distance between either x_1 <-> x_2 or x_2 <-> x_3
x = smp.symbols('x', real = True) # Free variable
err_y1, err_y2, err_y3, err_h = smp.symbols('E_1 E_2 E_3 E_h', real = True, constant = True) # ABSOLUTE errors of y_1, y_2, y_3, h
err_op1, err_op2, err_op3, err_op4, err_op5 = smp.symbols('\\epsilon_{op1}, \\epsilon_{op2} , \\epsilon_{op3}, \\epsilon_{op4}, \\epsilon_{op5}', real = True, constant = True) # RELATIVE algorithmic errors

First, we need to compute the parabola. We need to solve the system:

\begin{cases}
    a \, x_1 ^ 2 + b \, x_1 + c = y_1 \\
    a \, x_2 ^ 2 + b \, x_2 + c = y_2 \\
    a \, x_3 ^ 2 + b \, x_3 + c = y_3 \\
\end{cases}

In matrix form:

$$
    \begin{pmatrix}
        x_1^2 & x_1 & 1 \\
        x_2^2 & x_2 & 1 \\
        x_3^2 & x_3 & 1
    \end{pmatrix}
    \begin{pmatrix}
        a \\
        b \\
        c
    \end{pmatrix}
    =
    \begin{pmatrix}
        y_1 \\
        y_2 \\
        y_3
    \end{pmatrix}
$$

Known as the Vandermonde system.

In [3]:
A = smp.Matrix([ # Vandermonde matrix
    [x1 ** 2, x1, 1],
    [x2 ** 2, x2, 1],
    [x3 ** 2, x3, 1]
    ])

b_vect = smp.Matrix([ # Known vector
    y1,
    y2,
    y3
    ])

In [4]:
# Solution (a, b, c)
sol = A.LUsolve(b_vect)
sol = sol.applyfunc(lambda j: smp.factor(j))
sol

Matrix([
[                                         -(x_1*y_2 - x_1*y_3 - x_2*y_1 + x_2*y_3 + x_3*y_1 - x_3*y_2)/((x_1 - x_2)*(x_1 - x_3)*(x_2 - x_3))],
[                        (x_1**2*y_2 - x_1**2*y_3 - x_2**2*y_1 + x_2**2*y_3 + x_3**2*y_1 - x_3**2*y_2)/((x_1 - x_2)*(x_1 - x_3)*(x_2 - x_3))],
[(x_1**2*x_2*y_3 - x_1**2*x_3*y_2 - x_1*x_2**2*y_3 + x_1*x_3**2*y_2 + x_2**2*x_3*y_1 - x_2*x_3**2*y_1)/((x_1 - x_2)*(x_1 - x_3)*(x_2 - x_3))]])

Let the points are equally spaced, so $x_2 - x_1 = x_3 - x_2 = h$ (positive). The formulas are drastically simplified as shown:

In [5]:
sol = sol.applyfunc( # Introducing h. x2 is the middle point
    lambda j: j.subs([
        (x1, x2 - h),
        (x3, x2 + h)
        ]).factor()
        )

a, b, c = sol[0], sol[1].apart(h), sol[2].apart(h)
sol

Matrix([
[                                                          (y_1 - 2*y_2 + y_3)/(2*h**2)],
[                         -(h*y_1 - h*y_3 + 2*x_2*y_1 - 4*x_2*y_2 + 2*x_2*y_3)/(2*h**2)],
[(2*h**2*y_2 + h*x_2*y_1 - h*x_2*y_3 + x_2**2*y_1 - 2*x_2**2*y_2 + x_2**2*y_3)/(2*h**2)]])

In [6]:
a

(y_1 - 2*y_2 + y_3)/(2*h**2)

In [7]:
b

-(y_1 - y_3)/(2*h) - x_2*(y_1 - 2*y_2 + y_3)/h**2

In [8]:
c

y_2 + x_2*(y_1 - y_3)/(2*h) + x_2**2*(y_1 - 2*y_2 + y_3)/(2*h**2)

The parabola's equation is $y = a \, x^2 + b \, x + c$. Let's compute the area under the curve

In [9]:
integral = smp.integrate(a * x ** 2 + b * x + c, (x, x2 - h, x2 + h)).simplify() # Integral (Simpson's rule for a triplet)
integral

h*(y_1 + 4*y_2 + y_3)/3

Let's finally compute the numerical error committed. We start from the inherited error. If we consider the integral $I$ as a function of $h, y_1, y_2, y_3$, the absolute error committed is:

$$
    E_{inherited} = \left| \frac{\partial I}{\partial h} \, E_h \right| + \sum_{j = 1}^{3} \, \left| \frac{\partial I}{\partial y_j} \, E_j \right|
$$

In [10]:
err1 = sum([smp.Abs(smp.diff(integral, variable, 1) * err_variable) for variable, err_variable in zip([h, y1, y2, y3], [err_h, err_y1, err_y2, err_y3])]) # Computing the absolute inherited error
err1 = smp.simplify(err1).collect(h / 3) # Simplifying
err1

h*(Abs(E_1) + 4*Abs(E_2) + Abs(E_3))/3 + Abs(E_h*(y_1 + 4*y_2 + y_3))/3

The upper bound of the inherited error is:

$$
    E_{inherited} \leq \frac{2}{3} h u \, \left( |y_1| + 4 |y_2| + |y_3| \right)
$$

Where $u$ is one-half of machine precision. For doubles, $u \approx 1.1 \times 10^{-16}$

Let's compute the algorithmic error. We define the "fl()" operator:

$$
    \text{fl}(z) = z \, (1 + \epsilon_z)
$$

Where $\epsilon_z$ is the relative error committed by the machine when representing a real number. It's bounded by $u$:

$$
    | \varepsilon_z | \leq u
$$

In [ ]:
def fl(z, err_z): # fl() operator
    return z * (1 + err_z)

In [12]:
# Algorithm
op1 = fl(4 * y2, err_op1)
op2 = fl(y1 + op1, err_op2)
op3 = fl(op2 + y3, err_op3)
op4 = fl(h * op3, err_op4)
op5 = fl(op4 / 3, err_op5)

op5 # Final result

h*(\epsilon_{op3} + 1)*(\epsilon_{op4} + 1)*(\epsilon_{op5} + 1)*(y_3 + (\epsilon_{op2} + 1)*(y_1 + 4*y_2*(\epsilon_{op1} + 1)))/3

In [13]:
err_vars = [err_op1, err_op2, err_op3, err_op4, err_op5]

# First-order linearisation around err_op = 0
linear_part = 0
for e_var in err_vars:
    linear_part += smp.diff(op5, e_var).subs({e: 0 for e in err_vars}) * e_var

err2 = smp.Abs(smp.expand(linear_part).collect([y1, 4 * y2, y3]).collect(h / 3))
err2

h*Abs(y_1*(\epsilon_{op2} + \epsilon_{op3} + \epsilon_{op4} + \epsilon_{op5}) + 4*y_2*(\epsilon_{op1} + \epsilon_{op2} + \epsilon_{op3} + \epsilon_{op4} + \epsilon_{op5}) + y_3*(\epsilon_{op3} + \epsilon_{op4} + \epsilon_{op5}))/3

The upper bound of this quantity is:

$$
    E_{algo} \leq \frac{1}{3} h u \, \left( 4 |y_1| + 20 |y_2| + 3 |y_3| \right)
$$

A simple upper bound for the total error can be:

$$
    E_{tot} \leq 10 h u \left( |y_1| + |y_2| + |y_3| \right)
$$